# Data Preparation for Linear Probing Analysis

## Overview
This notebook prepares multilingual datasets for probing the internal representations of the Tiny Aya model across 13 diverse languages. We create two complementary probing tasks:

1. **Language Identification Probing**: Using FLORES+ parallel corpus to test if the model can distinguish between languages
2. **Part-of-Speech Tagging Probing**: Using Universal Dependencies treebanks to test syntactic understanding

## Target Languages
- **High-resource**: English, Spanish, French, German  
- **Medium-resource**: Arabic, Hindi, Bengali*, Tamil, Turkish, Persian
- **Low-resource**: Swahili*, Amharic, Yoruba

*Note: Bengali uses UD Bengali-BRU/PUD treebanks (with GitHub fallback). Swahili uses MasakhaPOS dataset as alternative to UD.

## Datasets
- **FLORES+**: 500 samples per language for language identification
- **Universal Dependencies**: 100 sentences per language for POS tagging (primary source)
- **Alternative Sources**: 
  - Bengali: UD Bengali-BRU/PUD treebanks (with GitHub fallback if needed)
  - Swahili: MasakhaPOS dataset from Masakhane

The prepared data will be used to train linear probes on different layers of Tiny Aya to understand how linguistic knowledge emerges across the model's depth.



In [35]:
_DATASET_DIR = "../datasets"

In [36]:
# Load HF_TOKEN from .env file
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    print("HF_TOKEN successfully loaded from .env file.")
else:
    print("HF_TOKEN not found in .env file. Proceeding without authentication.")

HF_TOKEN successfully loaded from .env file.


In [37]:
# Language family and script mapping
# Note: Bengali and Swahili removed from POS task due to unavailability in UD
language_metadata = {
    'English': {'script': 'Latin', 'family': 'Indo-European', 'resource_level': 'High'},
    'Spanish': {'script': 'Latin', 'family': 'Indo-European', 'resource_level': 'High'},
    'French': {'script': 'Latin', 'family': 'Indo-European', 'resource_level': 'High'},
    'German': {'script': 'Latin', 'family': 'Indo-European', 'resource_level': 'High'},
    'Arabic': {'script': 'Arabic', 'family': 'Afro-Asiatic', 'resource_level': 'Medium'},
    'Hindi': {'script': 'Devanagari', 'family': 'Indo-European', 'resource_level': 'Medium'},
    'Bengali': {'script': 'Bengali', 'family': 'Indo-European', 'resource_level': 'Medium'},  # Available for language_id only
    'Tamil': {'script': 'Tamil', 'family': 'Dravidian', 'resource_level': 'Medium'},
    'Turkish': {'script': 'Latin', 'family': 'Turkic', 'resource_level': 'Medium'},
    'Persian': {'script': 'Arabic', 'family': 'Indo-European', 'resource_level': 'Medium'},
    'Swahili': {'script': 'Latin', 'family': 'Niger-Congo', 'resource_level': 'Low'},  # Available for language_id only
    'Amharic': {'script': 'Ethiopic', 'family': 'Afro-Asiatic', 'resource_level': 'Low'},
    'Yoruba': {'script': 'Latin', 'family': 'Niger-Congo', 'resource_level': 'Low'}
}

# Create short language code for label
lang_code_mapping = {
    'English': 'en', 'Spanish': 'es', 'French': 'fr', 'German': 'de',
    'Arabic': 'ar', 'Hindi': 'hi', 'Bengali': 'bn', 'Tamil': 'ta',
    'Turkish': 'tr', 'Persian': 'fa', 'Swahili': 'sw', 'Amharic': 'am', 'Yoruba': 'yo'
}

In [38]:
import json
from datasets import load_dataset
from collections import Counter
from tqdm import tqdm


# Configuration: Mapping your 13 languages to FLORES+ configs
# format: { 'human_readable_name': 'flores_plus_config_name' }
languages = {
    'English': 'eng_Latn',
    'Spanish': 'spa_Latn',
    'French': 'fra_Latn',
    'German': 'deu_Latn',
    'Arabic': 'arb_Arab',
    'Hindi': 'hin_Deva',
    'Bengali': 'ben_Beng',
    'Tamil': 'tam_Taml',
    'Turkish': 'tur_Latn',
    'Persian': 'pes_Arab',
    'Swahili': 'swh_Latn',
    'Amharic': 'amh_Ethi',
    'Yoruba': 'yor_Latn'
}

SAMPLES_PER_LANG = 500
output_data = []


for display_name, config in languages.items():
    print(f"Loading dataset for {display_name} ({config})...")
    
    try:
        dataset = load_dataset("openlanguagedata/flores_plus", config, split='dev', trust_remote_code=True)
        
        # Shuffle to ensure randomness for selected subset
        dataset = dataset.shuffle(seed=42)
        
        # Take exactly 500 samples
        subset = dataset.select(range(min(SAMPLES_PER_LANG, len(dataset))))
        
        for index, item in enumerate(tqdm(subset)):
            output_data.append({
                "id": f"{lang_code_mapping[display_name]}_{index + 1}",
                "text": item['text'],
                "language": display_name.lower(),
                "label": lang_code_mapping[display_name],
                "task": "language_id",
                "metadata": language_metadata.get(display_name, {'script': 'Unknown', 'family': 'Unknown', 'resource_level': 'Unknown'})
            })
            
    except Exception as e:
        print(f"❌ Error loading {display_name}: {e}")



print(f"\n Done! Prepared {len(output_data)} examples.")
print(f"Stats: {dict(Counter(d['label'] for d in output_data))}")

Loading dataset for English (eng_Latn)...


100%|██████████| 500/500 [00:00<00:00, 15665.12it/s]


Loading dataset for Spanish (spa_Latn)...


100%|██████████| 500/500 [00:00<00:00, 16345.82it/s]


Loading dataset for French (fra_Latn)...


...

Loading dataset for Amharic (amh_Ethi)...


100%|██████████| 500/500 [00:00<00:00, 15755.50it/s]


Loading dataset for Yoruba (yor_Latn)...


100%|██████████| 500/500 [00:00<00:00, 15353.07it/s]


 Done! Prepared 6500 examples.
Stats: {'en': 500, 'es': 500, 'fr': 500, 'de': 500, 'ar': 500, 'hi': 500, 'bn': 500, 'ta': 500, 'tr': 500, 'fa': 500, 'sw': 500, 'am': 500, 'yo': 500}


In [39]:
output_data[0]

{'id': 'en_1',
 'text': 'Nevertheless, all French-speaking Belgians and Swiss would have learned standard French in school, so they would be able to understand you even if you used the standard French numbering system.',
 'language': 'english',
 'label': 'en',
 'task': 'language_id',
 'metadata': {'script': 'Latin',
  'family': 'Indo-European',
  'resource_level': 'High'}}

In [40]:
# Save to JSON
import os
os.makedirs(_DATASET_DIR, exist_ok=True)

with open(os.path.join(_DATASET_DIR, 'language_id_probing_data.json'), 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

In [41]:
# CORRECT Universal POS (UPOS) tags mapping from HuggingFace UD dataset
# This is the EXACT order used in the dataset's ClassLabel feature
UPOS_TAGS = [
    "NOUN",    # 0
    "PUNCT",   # 1
    "ADP",     # 2
    "NUM",     # 3
    "SYM",     # 4
    "SCONJ",   # 5
    "ADJ",     # 6
    "PART",    # 7
    "DET",     # 8
    "CCONJ",   # 9
    "PROPN",   # 10
    "PRON",    # 11
    "X",       # 12
    "_",       # 13 (unknown/missing)
    "ADV",     # 14
    "INTJ",    # 15
    "VERB",    # 16
    "AUX",     # 17
]

print("✅ UPOS tag mapping verified against HuggingFace UD dataset source code")

# Test the mapping with a sample
try:
    test_dataset = load_dataset("universal_dependencies", "en_gum", split='train[:1]', trust_remote_code=True)
    sample = test_dataset[0]
    print(f"\nVerification with English sample:")
    print(f"Tokens: {sample['tokens'][:5]}")
    print(f"UPOS IDs: {sample['upos'][:5]}")
    print(f"Mapped UPOS: {[UPOS_TAGS[i] if i < len(UPOS_TAGS) else 'UNK' for i in sample['upos'][:5]]}")
except Exception as e:
    print(f"⚠️  Could not verify mapping: {e}")


✅ UPOS tag mapping verified against HuggingFace UD dataset source code

Verification with English sample:
Tokens: ['Aesthetic', 'Appreciation', 'and', 'Spanish', 'Art']
UPOS IDs: [6, 0, 9, 6, 0]
Mapped UPOS: ['ADJ', 'NOUN', 'CCONJ', 'ADJ', 'NOUN']


In [42]:
import json
from datasets import load_dataset
from collections import Counter
from tqdm import tqdm 

# Primary UD Treebank Configs
ud_configs = {
    'English': ('en_gum', 'train'),
    'Spanish': ('es_ancora', 'train'),
    'French': ('fr_gsd', 'train'),
    'German': ('de_gsd', 'train'),
    'Arabic': ('ar_padt', 'train'),
    'Hindi': ('hi_hdtb', 'train'),
    'Tamil': ('ta_ttb', 'train'),
    'Turkish': ('tr_boun', 'train'),
    'Persian': ('fa_seraji', 'train'),
    'Amharic': ('am_att', 'test'),  # Only test split available
    'Yoruba': ('yo_ytb', 'test')    # Only test split available
}


SAMPLES_PER_LANG = 100
pos_data = []

print("Starting POS Dataset Preparation...")

for display_name, (config, split) in ud_configs.items():
    print(f"Loading UD {display_name} ({config})...")
    try:
        # Load the dataset with the appropriate split
        dataset = load_dataset("universal_dependencies", config, split=split, trust_remote_code=True)
        dataset = dataset.shuffle(seed=42)
        
        subset = dataset.select(range(min(SAMPLES_PER_LANG, len(dataset))))
        
        for index, item in enumerate(tqdm(subset)):
            # item['tokens'] is a list of words
            # item['upos'] is a list of integer IDs for POS tags
            # Map POS IDs to actual POS tag strings
            pos_labels = [UPOS_TAGS[pos_id] if 0 <= pos_id < len(UPOS_TAGS) else "X" for pos_id in item['upos']]
            
            
            pos_data.append({
                "id":  f"{lang_code_mapping[display_name]}_{index + 1}", # Sequential ID
                "language": display_name.lower(),
                "label": lang_code_mapping[display_name],
                "text": item['text'],
                "tokens": item['tokens'],
                "pos_tags": pos_labels,
                "task": "pos_tagging",
                "metadata": language_metadata.get(display_name, {'script': 'Unknown', 'family': 'Unknown', 'resource_level': 'Unknown'})
            })
            
    except Exception as e:
        print(f"❌ Error loading {display_name}: {e}")


print(f"\nDone! Prepared {len(pos_data)} sentences for POS probing.")

Starting POS Dataset Preparation...
Loading UD English (en_gum)...


100%|██████████| 100/100 [00:00<00:00, 6261.00it/s]


Loading UD Spanish (es_ancora)...


100%|██████████| 100/100 [00:00<00:00, 759.11it/s]


Loading UD French (fr_gsd)...


...

Loading UD Amharic (am_att)...


100%|██████████| 100/100 [00:00<00:00, 5853.47it/s]


Loading UD Yoruba (yo_ytb)...


100%|██████████| 100/100 [00:00<00:00, 4762.84it/s]


Done! Prepared 1100 sentences for POS probing.


In [43]:
# Save to JSON
import os
os.makedirs(_DATASET_DIR, exist_ok=True)

with open(os.path.join(_DATASET_DIR, 'pos_probing_data.json'), 'w', encoding='utf-8') as f:
    json.dump(pos_data, f, ensure_ascii=False, indent=2)